# Lesson 8.2 — Performance Architecture: Caching & Purpose-Built Databases

**Level: Advanced** | **Exam:** SAP-C02 Domain 2, Task 2.5

## Performance Architecture Principles

### Caching Strategy
```
Request → Application → Cache HIT? → Return cached data (fast)
                     → Cache MISS? → Database → Store in cache → Return data
```

### AWS Caching Services
| Service | Use Case | TTL |
|---------|---------|-----|
| **ElastiCache Redis** | Session, leaderboards, real-time analytics | Seconds to forever |
| **ElastiCache Memcached** | Simple object caching, no persistence | Seconds to days |
| **CloudFront** | Static content, API responses | Minutes to years |
| **DAX (DynamoDB Accelerator)** | DynamoDB read offloading (microseconds) | 5 min (default) |
| **RDS Read Replicas** | Read-heavy workloads | Real-time |

### Purpose-Built Database Selection
| Need | AWS Service |
|------|------------|
| Relational, ACID, complex queries | Aurora (MySQL/PostgreSQL) |
| Key-value at scale, single-digit ms | DynamoDB |
| In-memory, sub-ms | ElastiCache Redis / MemoryDB |
| Search (full-text) | OpenSearch Service |
| Graph relationships | Neptune |
| Time-series metrics | Timestream |
| Data warehouse | Redshift |
| Document store | DocumentDB |
| Wide column | Keyspaces (Cassandra) |

### DynamoDB Design Patterns
- **Partition key must distribute evenly** — hot partition = throttling
- Use **DAX** for read-heavy applications (microsecond vs millisecond reads)
- **Global Tables** for multi-region active/active
- **DynamoDB Streams** for event-driven architectures

In [ ]:
import boto3, json
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# Explore ElastiCache and DAX
import boto3, json

elasticache = boto3.client('elasticache', region_name='us-east-1')
dax = boto3.client('dax', region_name='us-east-1')

# List ElastiCache clusters
try:
    redis = elasticache.describe_replication_groups()['ReplicationGroups']
    print(f"ElastiCache Redis Replication Groups ({len(redis)}):")
    for rg in redis:
        node_count = len(rg.get('NodeGroups', [{}])[0].get('NodeGroupMembers', []))
        multi_az = rg.get('MultiAZ', 'disabled')
        print(f"  {rg['ReplicationGroupId']:30} {rg['Status']:12} Nodes: {node_count} MultiAZ: {multi_az}")
    if not redis:
        print("  No Redis clusters found.")
except Exception as e:
    print(f"ElastiCache: {e}")

# List DAX clusters
try:
    clusters = dax.describe_clusters()['Clusters']
    print(f"\nDAX Clusters ({len(clusters)}):")
    for cluster in clusters:
        print(f"  {cluster['ClusterName']:30} {cluster['Status']:12} Nodes: {cluster['TotalNodes']}")
    if not clusters:
        print("  No DAX clusters found.")
except Exception as e:
    print(f"DAX: {e}")

# Database selection guide
print("\n=== Purpose-Built Database Selection ===")
use_cases = [
    ("E-commerce product catalog (flexible schema)", "DynamoDB"),
    ("Financial transactions (ACID compliance)", "Aurora PostgreSQL"),
    ("Social network (friend relationships)", "Neptune"),
    ("Real-time analytics dashboard", "ElastiCache Redis"),
    ("Full-text search for documents", "OpenSearch Service"),
    ("IoT sensor readings over time", "Timestream"),
    ("BI and data warehouse", "Redshift"),
]
for use_case, service in use_cases:
    print(f"  {use_case:55} → {service}")

## Exercise

In [ ]:
# TODO: Design a caching strategy for a high-traffic e-commerce platform
# The platform has:
# - Product catalog (10M products, read 100x more than written)
# - User sessions (millions of concurrent users)
# - Shopping cart (updated frequently per user)
# - Order history (infrequent reads, write-once)
# - Real-time inventory (must be accurate, high contention)
#
# For each data type, specify:
# - Which AWS caching/database service to use
# - TTL strategy
# - Cache invalidation approach

data_strategy = {
    "product_catalog": {
        "primary_db": "TODO",
        "cache_service": "TODO",
        "ttl": "TODO",
        "invalidation": "TODO"
    },
    "user_sessions": {
        "service": "TODO",
        "ttl": "TODO",
        "reason": "TODO"
    },
    "shopping_cart": {
        "service": "TODO",
        "reason": "TODO"
    },
    "order_history": {
        "service": "TODO",
        "reason": "TODO"
    },
    "real_time_inventory": {
        "service": "TODO",
        "reason": "TODO"
    }
}

import json
print(json.dumps(data_strategy, indent=2))

## Summary
- **Cache-aside pattern**: App checks cache first, reads from DB on miss, writes to cache
- **ElastiCache Redis**: Versatile — sessions, leaderboards, pub/sub, rate limiting, Lua scripts
- **DAX**: Transparent caching layer for DynamoDB — microsecond reads, no app changes
- **Purpose-built databases** outperform general-purpose databases for specific use cases
- DynamoDB hot partition = evenly distribute partition keys; use write sharding if needed

---
**Next:** [Lesson 8.3 — Serverless & Containers: Lambda, ECS, EKS & Fargate](lesson-8.3-serverless-containers.ipynb)